In [3]:
from training.dataset_classes.text_datasets import TextClassificationDataset
import pandas as pd
import numpy as np
from pathlib import Path
from itertools import product
from glob import glob
from pathlib import Path

### 1. Create train dataset

In [4]:
# data_path = '/app/datasets/text_datasets/yahoo_answers_csv/in_distribution_train.csv'
# df = pd.read_csv(
#     data_path,           # or 'your_file.csv'
#     header=None,
#     usecols=[0, 2],           # keep only column 0 (first) and column 2 (third)
#     names=['class', 'text'],   # rename them as needed
#     quotechar='"'
# )
# df['text'] = df['text'].str.strip('"\'')
# # Re-check
# print("Remaining leading/trailing quotes:",
#       (df['text'].str.startswith('"') | df['text'].str.endswith('"') |
#        df['text'].str.startswith("'") | df['text'].str.endswith("'")).sum())
# unique_classes = sorted(np.unique(df['class'].values))
# class_to_idx = {cls: idx for idx, cls in enumerate(unique_classes)}
# df['class'] = df['class'].map(class_to_idx)

### 2. Create an OSR protocol for text

In [5]:
# data_path = "/app/datasets/text_datasets/yahoo_answers_csv/in_distribution_test.csv"
# df = pd.read_csv(
#     data_path,  # or 'your_file.csv'
#     header=None,
#     usecols=[0, 2],  # keep only column 0 (first) and column 2 (third)
#     names=["class", "text"],  # rename them as needed
#     quotechar='"',
# )
# df["text"] = df["text"].str.strip("\"'")

In [6]:
# np.unique(df["class"].values, return_counts=True)

##### Protocol construction
1. Each sample is its own template
2. I have to split in-distribution test set into gallery and probes.  
I will randomly choose 2% of in-distribution samples from each class to construct gallery templates. 
3. I will use out-of-disribution samples as it is

In [7]:
# np.where(df["class"].values == 2)[0]

In [8]:
# from training.dataset_classes.text_datasets import TextPredictionDataset
# ds = TextPredictionDataset('/app/datasets/text-ident/yahoo_answers')
# ds[1]

In [ ]:
ident_ds_dir = Path("/app/datasets/text-ident")
ident_ds_dir.mkdir(exist_ok=True)

# list protocols
protools_path = "/app/datasets/text_datasets"
protools_paths = list(glob(protools_path + "/*_csv"))

# create meta files
template_idx_shift = 10000  # shift to get unique template ids
gallery_template_size_fraction = 1e-2
min_gallery_template_size = 50
rng = np.random.default_rng(32)

for protocol_path in protools_paths:
    protocol_path = Path(protocol_path)
    print(protocol_path)
    ds_name = (protocol_path.parts[-1]).lower()[:-4]
    meta_path = ident_ds_dir / ds_name / "meta"
    meta_path.mkdir(exist_ok=True, parents=True)

    ossi_info_path = protocol_path
    df_in_disribution = pd.read_csv(
        ossi_info_path / "in_distribution_test.csv",
        header=None,
        usecols=[0, 2],
        names=["class", "text"],
        quotechar='"',
    )
    df_in_disribution["text"] = df_in_disribution["text"].str.strip("\"'")

    df_ood = pd.read_csv(
        ossi_info_path / "out_distribution_test.csv",
        header=None,
        usecols=[0, 2],
        names=["class", "text"],
        quotechar='"',
    )
    df_ood["text"] = df_ood["text"].str.strip("\"'")

    known_classes, known_count = np.unique(df_in_disribution["class"].values, return_counts=True)
    gallery_template_sizes = np.max(
        np.stack(
            [
                np.array([min_gallery_template_size] * known_classes.shape[0]),
                (
                    gallery_template_size_fraction
                    * known_count
                ).astype("int"),
            ],
            axis=1,
        ),
        axis=1,
    )
    # save texts into files
    known_save_dir = ident_ds_dir / ds_name / 'known_texts'
    known_save_dir.mkdir(exist_ok=True)
    for i, text in enumerate(df_in_disribution['text'].values):
        with open(known_save_dir / f'{i}.txt', 'w') as fd:
            fd.write(text)
    unknown_save_dir = ident_ds_dir / ds_name / 'unknown_texts'
    unknown_save_dir.mkdir(exist_ok=True)
    for i, text in enumerate(df_ood['text'].values):
        with open(unknown_save_dir / f'{i}.txt', 'w') as fd:
            fd.write(text)

    # select gallery and probe templates
    gallery_paths = []
    gallery_ids = []
    probe_paths = []
    probe_ids = []
    for i, known_class in enumerate(known_classes):
        sample_of_class = np.where(df_in_disribution["class"].values == known_class)[0]
        gallery_sample_idx = rng.choice(sample_of_class, size=gallery_template_sizes[i], replace=False)
        probe_sample_idx = set(sample_of_class) - set(gallery_sample_idx)
        gallery_paths.append([f'known_texts/{sample_id}.txt' for sample_id in gallery_sample_idx])
        gallery_ids.append(([known_class] * gallery_template_sizes[i]))
        probe_paths.append([f'known_texts/{sample_id}.txt' for sample_id in np.array(list(probe_sample_idx))])
        probe_ids.append([known_class] * len(probe_sample_idx))
    gallery_paths = np.concatenate(gallery_paths)
    gallery_ids = np.concatenate(gallery_ids)
    probe_paths = np.concatenate(probe_paths)
    probe_ids = np.concatenate(probe_ids)

    probe_paths = np.concatenate([probe_paths, [f'unknown_texts/{sample_id}.txt' for sample_id in range(len(df_ood['text'].values))]])
    probe_ids = np.concatenate([probe_ids, df_ood['class'].values])

    probe_template_ids = np.arange(len(probe_ids))+template_idx_shift
    # # create tid/mid file
    text_paths = np.concatenate([gallery_paths,probe_paths])
    ids = np.concatenate([gallery_ids, probe_ids])
    tids = np.concatenate([gallery_ids, probe_template_ids])
    mids = np.arange(len(ids))
    out_file_tid_mid = meta_path / Path(f"{ds_name}_face_tid_mid.txt")
    with open(out_file_tid_mid, "w") as fd:
        for name, tid, sid, mid in zip(text_paths, tids, ids, mids):
            fd.write(f"{name} {tid} {mid} {sid}\n")

    # # create gallery and probe meta files
    out_file_probe = meta_path / Path(f"{ds_name}_1N_probe_mixed.csv")
    out_file_gallery = meta_path / Path(f"{ds_name}_1N_gallery_G1.csv")

    assert len(gallery_ids) + len(probe_ids) == len(text_paths)
    probe = pd.DataFrame(
        {
            "TEMPLATE_ID": probe_template_ids,
            "SUBJECT_ID": probe_ids,
            "FILENAME": probe_paths,
        }
    )
    gallery = pd.DataFrame(
        {
            "TEMPLATE_ID": gallery_ids,
            "SUBJECT_ID": gallery_ids,
            "FILENAME": gallery_paths,
        }
    )

    probe.to_csv(out_file_probe, sep=",", index=False)
    gallery.to_csv(out_file_gallery, sep=",", index=False)

### Validation sets

In [10]:
# ident_ds_dir = Path("/app/datasets/text-ident-val")
# ident_ds_dir.mkdir(exist_ok=True)

# # list protocols
# protools_path = "/app/datasets/text_datasets"
# protools_paths = list(glob(protools_path + "/*_csv"))

# # create meta files
# template_idx_shift = 10000  # shift to get unique template ids
# gallery_template_size_fraction = 1e-2
# min_gallery_template_size = 50
# rng = np.random.default_rng(32)

# for protocol_path in protools_paths:
#     protocol_path = Path(protocol_path)
#     print(protocol_path)
#     ds_name = (protocol_path.parts[-1]).lower()[:-4]
#     meta_path = ident_ds_dir / ds_name / "meta"
#     meta_path.mkdir(exist_ok=True, parents=True)

#     ossi_info_path = protocol_path
#     df_in_disribution = pd.read_csv(
#         ossi_info_path / "in_distribution_train.csv",
#         header=None,
#         usecols=[0, 2],
#         names=["class", "text"],
#         quotechar='"',
#     )
#     df_in_disribution_test = pd.read_csv(
#         ossi_info_path / "in_distribution_test.csv",
#         header=None,
#         usecols=[0, 2],
#         names=["class", "text"],
#         quotechar='"',
#     )
#     df_in_disribution["text"] = df_in_disribution["text"].str.strip("\"'")
#     df_ood = pd.read_csv(
#         ossi_info_path / "out_distribution_train.csv",
#         header=None,
#         usecols=[0, 2],
#         names=["class", "text"],
#         quotechar='"',
#     )
#     df_ood_test = pd.read_csv(
#         ossi_info_path / "out_distribution_test.csv",
#         header=None,
#         usecols=[0, 2],
#         names=["class", "text"],
#         quotechar='"',
#     )
#     df_ood["text"] = df_ood["text"].str.strip("\"'")

#     known_classes, known_count_test = np.unique(df_in_disribution_test["class"].values, return_counts=True)
#     gallery_template_sizes = np.max(
#         np.stack(
#             [
#                 np.array([min_gallery_template_size] * known_classes.shape[0]),
#                 (
#                     gallery_template_size_fraction
#                     * known_count_test
#                 ).astype("int"),
#             ],
#             axis=1,
#         ),
#         axis=1,
#     )
#     # save texts into files
#     known_save_dir = ident_ds_dir / ds_name / 'known_texts'
#     known_save_dir.mkdir(exist_ok=True)


#     ood_texts = df_ood['text'].values
#     ood_class = df_ood['class'].values
#     ood_ids = rng.choice(np.arange(len(ood_texts)), size=len(df_ood_test['text'].values), replace=False)
#     unknown_save_dir = ident_ds_dir / ds_name / 'unknown_texts'
#     unknown_save_dir.mkdir(exist_ok=True)
#     for i, text in enumerate(ood_texts[ood_ids]):
#         with open(unknown_save_dir / f'{i}.txt', 'w') as fd:
#             fd.write(text)

#     # select gallery and probe templates
#     gallery_paths = []
#     gallery_ids = []
#     probe_paths = []
#     probe_ids = []
#     for i, known_class in enumerate(known_classes):
#         sample_of_class = np.where(df_in_disribution["class"].values == known_class)[0]
#         # sample the same number of in distributions sample as in test
#         sample_of_class = rng.choice(sample_of_class, known_count_test[i], replace=False)
#         for j, text in zip(sample_of_class, df_in_disribution['text'].values[sample_of_class]):
#             with open(known_save_dir / f'{j}.txt', 'w') as fd:
#                 fd.write(text)
#         gallery_sample_idx = rng.choice(sample_of_class, size=gallery_template_sizes[i], replace=False)
#         probe_sample_idx = set(sample_of_class) - set(gallery_sample_idx)
#         gallery_paths.append([f'known_texts/{sample_id}.txt' for sample_id in gallery_sample_idx])
#         gallery_ids.append(([known_class] * gallery_template_sizes[i]))
#         probe_paths.append([f'known_texts/{sample_id}.txt' for sample_id in np.array(list(probe_sample_idx))])
#         probe_ids.append([known_class] * len(probe_sample_idx))
#     gallery_paths = np.concatenate(gallery_paths)
#     gallery_ids = np.concatenate(gallery_ids)
#     probe_paths = np.concatenate(probe_paths)
#     probe_ids = np.concatenate(probe_ids)


#     probe_paths = np.concatenate([probe_paths, [f'unknown_texts/{sample_id}.txt' for sample_id in range(len(ood_ids))]])
#     probe_ids = np.concatenate([probe_ids, ood_class[ood_ids]])

#     probe_template_ids = np.arange(len(probe_ids))+template_idx_shift
#     # # create tid/mid file
#     text_paths = np.concatenate([gallery_paths,probe_paths])
#     ids = np.concatenate([gallery_ids, probe_ids])
#     tids = np.concatenate([gallery_ids, probe_template_ids])
#     mids = np.arange(len(ids))
#     out_file_tid_mid = meta_path / Path(f"{ds_name}_face_tid_mid.txt")
#     with open(out_file_tid_mid, "w") as fd:
#         for name, tid, sid, mid in zip(text_paths, tids, ids, mids):
#             fd.write(f"{name} {tid} {mid} {sid}\n")

#     # # create gallery and probe meta files
#     out_file_probe = meta_path / Path(f"{ds_name}_1N_probe_mixed.csv")
#     out_file_gallery = meta_path / Path(f"{ds_name}_1N_gallery_G1.csv")

#     assert len(gallery_ids) + len(probe_ids) == len(text_paths)
#     probe = pd.DataFrame(
#         {
#             "TEMPLATE_ID": probe_template_ids,
#             "SUBJECT_ID": probe_ids,
#             "FILENAME": probe_paths,
#         }
#     )
#     gallery = pd.DataFrame(
#         {
#             "TEMPLATE_ID": gallery_ids,
#             "SUBJECT_ID": gallery_ids,
#             "FILENAME": gallery_paths,
#         }
#     )

#     probe.to_csv(out_file_probe, sep=",", index=False)
#     gallery.to_csv(out_file_gallery, sep=",", index=False)

In [12]:
import shutil

# copy embeddings
dataset_names = ["yahoo_answers", "agnews", "dbpedia"]
for name in dataset_names:
    # val
    embeddings_dir_val = (
        Path("/app/datasets/text-ident-val") / f"{name}-val" / "embeddings"
    )
    embeddings_dir_val.mkdir(exist_ok=True)
    file_path = Path(f"/app/cache/features/scf_2epoch_topic_{name}_val_embs.npz")
    if file_path.is_file():
        shutil.copyfile(file_path, embeddings_dir_val / f"scf_embs_{name}.npz")
    # test
    embeddings_dir_test = Path("/app/datasets/text-ident") / f"{name}" / "embeddings"
    embeddings_dir_test.mkdir(exist_ok=True)
    file_path = Path(f"/app/cache/features/scf_2epoch_topic_{name}_test_embs.npz")
    if file_path.is_file():
        shutil.copyfile(file_path, embeddings_dir_test / f"scf_embs_{name}.npz")

## Clinc150 (use embedding train set as the gallery)

In [ ]:
# from training.dataset_classes.text_datasets import Clinc150DataModule

# clinc150_dataset = Clinc150DataModule('/app/datasets/clinc150/data_full.json')
# clinc150_dataset.setup()

# ds_type = 'val'
# ident_ds_dir = Path(f"/app/datasets/clinc150_{ds_type}")
# ident_ds_dir.mkdir(exist_ok=True)

# # create meta files
# template_idx_shift = 10000  # shift to get unique template ids
# rng = np.random.default_rng(32)

# meta_path = ident_ds_dir / "meta"
# meta_path.mkdir(exist_ok=True, parents=True)

# # init ds
# if ds_type == 'val':
#     ds = clinc150_dataset.val_dataset
# elif ds_type == 'test':
#     ds = clinc150_dataset.test_dataset

# # save texts into files
# # first save train texts, then val/test
# text_save_dir = ident_ds_dir / 'texts'
# text_save_dir.mkdir(exist_ok=True)

# text_counter = 0
# gallery_paths = []
# gallery_ids = []
# probe_paths = []
# probe_ids = []
# for i in range(len(clinc150_dataset.train_dataset)):
#     gallery_ids.append(clinc150_dataset.train_dataset[i]['label'])
#     gallery_paths.append(f'texts/{text_counter}.txt')
#     with open(text_save_dir / f'{text_counter}.txt', 'w') as fd:
#         fd.write(clinc150_dataset.train_dataset[i]['text'])
#     text_counter+=1

# for i in range(len(ds)):
#     probe_ids.append(ds[i]['label'])
#     probe_paths.append(f'texts/{text_counter}.txt')
#     with open(text_save_dir / f'{text_counter}.txt', 'w') as fd:
#         fd.write(ds[i]['text'])
#     text_counter+=1

# # select gallery and probe templates

# gallery_paths = np.array(gallery_paths)
# gallery_ids = np.array(gallery_ids)
# probe_paths = np.array(probe_paths)
# probe_ids = np.array(probe_ids)


# probe_template_ids = np.arange(len(probe_ids)) + template_idx_shift
# # # create tid/mid file
# text_paths = np.concatenate([gallery_paths,probe_paths])
# ids = np.concatenate([gallery_ids, probe_ids])
# tids = np.concatenate([gallery_ids, probe_template_ids])
# mids = np.arange(len(ids))
# if ds_type == 'val':
#     name_appendix = "_val"
# else:
#     name_appendix = ""
# out_file_tid_mid = meta_path / Path(f"clinc150{name_appendix}_face_tid_mid.txt")
# with open(out_file_tid_mid, "w") as fd:
#     for name, tid, sid, mid in zip(text_paths, tids, ids, mids):
#         fd.write(f"{name} {tid} {mid} {sid}\n")

# # # create gallery and probe meta files
# out_file_probe = meta_path / Path(f"clinc150{name_appendix}_1N_probe_mixed.csv")
# out_file_gallery = meta_path / Path(f"clinc150{name_appendix}_1N_gallery_G1.csv")

# assert len(gallery_ids) + len(probe_ids) == len(text_paths)
# probe = pd.DataFrame(
#     {
#         "TEMPLATE_ID": probe_template_ids,
#         "SUBJECT_ID": probe_ids,
#         "FILENAME": probe_paths,
#     }
# )
# gallery = pd.DataFrame(
#     {
#         "TEMPLATE_ID": gallery_ids,
#         "SUBJECT_ID": gallery_ids,
#         "FILENAME": gallery_paths,
#     }
# )

# probe.to_csv(out_file_probe, sep=",", index=False)
# gallery.to_csv(out_file_gallery, sep=",", index=False)

In [ ]:
# # copy_embs
# val_embs_dir = Path(f"/app/datasets/clinc150_val") / 'embeddings'
# val_embs_dir.mkdir(exist_ok=True)

# test_embs_dir = Path(f"/app/datasets/clinc150_test") / 'embeddings'
# test_embs_dir.mkdir(exist_ok=True)

# train_embs = np.load('/app/cache/features/clinc150_train_embs.npz')
# val_embs = np.load('/app/cache/features/clinc150_val_embs.npz')
# test_embs = np.load('/app/cache/features/clinc150_test_embs.npz')

# np.savez(val_embs_dir / 'scf_embs_clinc150_val.npz', embs=np.concatenate([train_embs['embs'], val_embs['embs']], axis=0),
#          unc = np.concatenate([train_embs['unc'], val_embs['unc']], axis=0))
# np.savez(test_embs_dir / 'scf_embs_clinc150.npz', embs=np.concatenate([train_embs['embs'], test_embs['embs']], axis=0),
#          unc = np.concatenate([train_embs['unc'], test_embs['unc']], axis=0))

In [ ]:
# train_embs['embs'].shape, train_embs['unc'].shape

### PAN Authorship indentification

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from typing import Dict, List, Tuple


def build_osr_protocol_exact_order(
    datamodule,
    output_dir: str,
    ds_type: str = "val",
    gallery_docs_per_author: int = 3,
    template_idx_shift: int = 10000,
    ds_name: str = "pan",
    use_one_sample_per_template: bool = False,
):
    """
    Construct OSR protocol with STRICT index alignment:
      Line N in tid_mid.txt  ↔  texts/N.txt  ↔  embeddings.npz[N]

    Order: [gallery_samples (3 per author)] + [probe_samples (val_dataset order)]
    """

    rng = np.random.default_rng(0)

    output_path = Path(output_dir)
    output_path.mkdir(exist_ok=True, parents=True)
    text_save_dir = output_path / "texts"
    text_save_dir.mkdir(exist_ok=True)
    meta_path = output_path / "meta"
    meta_path.mkdir(exist_ok=True)

    # Step 1: Determine source JSONL and author lists based on ds_type
    if ds_type == "val":
        gallery_authors = datamodule.val_in_gallery
        out_of_gallery_authors = datamodule.val_out_gallery
        dataset = datamodule.val_dataset
    elif ds_type == "test":
        gallery_authors = datamodule.test_in_gallery
        out_of_gallery_authors = datamodule.test_out_gallery
        dataset = datamodule.test_dataset
    else:
        raise ValueError(f"Unknown ds_type: {ds_type}")
    index_to_meta = defaultdict(list)

    author_to_ids = defaultdict(list)
    for i in range(len(dataset)):
        author_to_ids[dataset[i]["author_id"]].append(i)
        # create text entry
        with open(text_save_dir / f"{i}.txt", "w") as fd:
            fd.write(dataset[i]["text"])
    author_id = 0
    media_id = 0
    probe_template_id = 0

    probe_template_ids = []
    probe_ids = []
    probe_paths = []
    gallery_paths = []
    gallery_ids = []

    for gallery_author in gallery_authors:
        in_gallery_samples = rng.choice(
            author_to_ids[gallery_author], gallery_docs_per_author, replace=False
        )
        for in_gallery_sample in in_gallery_samples:
            # fd.write(f"{name} {tid} {mid} {sid}\n")
            name = f"texts/{in_gallery_sample}.txt"
            index_to_meta[in_gallery_sample] = [name, author_id, media_id, author_id]
            gallery_ids.append(author_id)
            gallery_paths.append(name)
            media_id += 1
        for probe_in_gallery_sample in set(author_to_ids[gallery_author]) - set(
            in_gallery_samples
        ):
            name = f"texts/{probe_in_gallery_sample}.txt"
            index_to_meta[probe_in_gallery_sample] = [
                name,
                probe_template_id + template_idx_shift,
                media_id,
                author_id,
            ]

            probe_template_ids.append(probe_template_id + template_idx_shift)
            probe_ids.append(author_id)
            probe_paths.append(name)
            media_id += 1
            if use_one_sample_per_template:
                probe_template_id += 1
        if not use_one_sample_per_template:
            probe_template_id += 1
        author_id += 1
    for out_of_gallery_author in out_of_gallery_authors:
        for oog_sample_id in author_to_ids[out_of_gallery_author]:
            name = f"texts/{oog_sample_id}.txt"
            index_to_meta[oog_sample_id] = [
                name,
                probe_template_id + template_idx_shift,
                media_id,
                author_id,
            ]

            probe_template_ids.append(probe_template_id + template_idx_shift)
            probe_ids.append(author_id)
            probe_paths.append(name)
            if use_one_sample_per_template:
                probe_template_id += 1
            media_id += 1
        author_id += 1
        if not use_one_sample_per_template:
            probe_template_id += 1

    out_file_tid_mid = meta_path / Path(f"{ds_name}_{ds_type}_face_tid_mid.txt")
    with open(out_file_tid_mid, "w") as fd:
        for i in range(len(dataset)):
            name, tid, mid, sid = index_to_meta[i]
            fd.write(f"{name} {tid} {mid} {sid}\n")

    # # create gallery and probe meta files
    out_file_probe = meta_path / Path(f"{ds_name}_{ds_type}_1N_probe_mixed.csv")
    out_file_gallery = meta_path / Path(f"{ds_name}_{ds_type}_1N_gallery_G1.csv")

    assert len(gallery_ids) + len(probe_ids) == len(dataset)
    probe = pd.DataFrame(
        {
            "TEMPLATE_ID": probe_template_ids,
            "SUBJECT_ID": probe_ids,
            "FILENAME": probe_paths,
        }
    )
    gallery = pd.DataFrame(
        {
            "TEMPLATE_ID": gallery_ids,
            "SUBJECT_ID": gallery_ids,
            "FILENAME": gallery_paths,
        }
    )

    probe.to_csv(out_file_probe, sep=",", index=False)
    gallery.to_csv(out_file_gallery, sep=",", index=False)

In [ ]:
# from training.dataset_classes.pan_text import PANDataModule

# dm = PANDataModule(
#     train_jsonl="/app/datasets/pan/unseen_authors/xl/pan20-av-large-notest.jsonl",
#     test_jsonl="/app/datasets/pan/unseen_authors/xl/pan20-av-large-test.jsonl",
#     batch_size=64,
#     num_workers=16,
#     tokenizer_name="bert-base-uncased",
#     max_length=512,
#     min_docs_per_author=10,  # MUST match training config
#     train_authors=4000,
#     val_authors=200,
#     val_probe_authors=200,
#     test_authors=200,
#     test_probe_authors=200,
# )
# dm.setup()

In [ ]:
# import shutil
# import os

# shutil.rmtree('/app/cache/template_cache_new/scf')
# use_one_sample_per_template = False
# ds_type = "val"
# outdir = Path(f"/app/datasets/pan_{ds_type}")
# os.remove(outdir / 'backup.npz')
# os.remove(outdir / 'gallery_prob_backup.npz')
# build_osr_protocol_exact_order(
#     datamodule=dm,
#     output_dir=outdir,
#     ds_type=ds_type,
#     gallery_docs_per_author=3,
#     template_idx_shift=10000,
#     use_one_sample_per_template = use_one_sample_per_template,
# )
# embeddings_dir = outdir / "embeddings"
# embeddings_dir.mkdir(exist_ok=True)
# shutil.copyfile("/app/cache/features/pan_val_embs.npz", embeddings_dir / "scf_embs_pan_val.npz")

# ds_type = "test"
# outdir = Path(f"/app/datasets/pan_{ds_type}")
# os.remove(outdir / 'backup.npz')
# os.remove(outdir / 'gallery_prob_backup.npz')
# build_osr_protocol_exact_order(
#     datamodule=dm,
#     output_dir=outdir,
#     ds_type=ds_type,
#     gallery_docs_per_author=3,
#     template_idx_shift=10000,
#     use_one_sample_per_template = use_one_sample_per_template,
# )
# embeddings_dir = outdir / "embeddings"
# embeddings_dir.mkdir(exist_ok=True)
# shutil.copyfile("/app/cache/features/pan_test_embs.npz", embeddings_dir / "scf_embs_pan_test.npz")

### Blog Authorship dataset

In [ ]:
from training.dataset_classes.blog_text import BlogAuthorshipDataModule
import shutil
import os

# shutil.rmtree('/app/cache/template_cache_new/scf')
dm_blog = BlogAuthorshipDataModule(
    csv_path="/app/datasets/blog_authorship_corpus/blogtext.csv",
    tokenizer_name="bert-base-uncased",
    batch_size=512,
    num_workers=16,
    max_length=512,
    min_docs_per_author=10,
    train_authors=4000,
    val_authors=1000,
    val_probe_authors=1000,
    test_authors=1000,
)
dm_blog.setup()

# shutil.rmtree('/app/cache/template_cache_new/scf')
use_one_sample_per_template = False

ds_type = "val"
outdir = Path(f"/app/datasets/blog_{ds_type}")
# os.remove(outdir / 'backup.npz')
# os.remove(outdir / 'gallery_prob_backup.npz')
build_osr_protocol_exact_order(
    datamodule=dm_blog,
    output_dir=outdir,
    ds_type=ds_type,
    gallery_docs_per_author=3,
    template_idx_shift=10000,
    ds_name="blog",
    use_one_sample_per_template=use_one_sample_per_template,
)
embeddings_dir = outdir / "embeddings"
embeddings_dir.mkdir(exist_ok=True)
shutil.copyfile(
    "/app/cache/features/blog_val_embs.npz", embeddings_dir / "scf_embs_blog_val.npz"
)


ds_type = "test"
outdir = Path(f"/app/datasets/blog_{ds_type}")
# os.remove(outdir / 'backup.npz')
# os.remove(outdir / 'gallery_prob_backup.npz')
build_osr_protocol_exact_order(
    datamodule=dm_blog,
    output_dir=outdir,
    ds_type=ds_type,
    gallery_docs_per_author=3,
    template_idx_shift=10000,
    ds_name="blog",
    use_one_sample_per_template=use_one_sample_per_template,
)
embeddings_dir = outdir / "embeddings"
embeddings_dir.mkdir(exist_ok=True)
shutil.copyfile(
    "/app/cache/features/blog_test_embs.npz", embeddings_dir / "scf_embs_blog_test.npz"
)

✓ Parsed 681284 rows → 19320 authors
[full] 635744 docs | 10403 authors | min_docs=10
[train] 248732 docs | 4000 authors | min_docs=10
[val] 60285 docs | 1000 authors | min_docs=10
[test] 53678 docs | 1000 authors | min_docs=10
✅ OSR integrity verified: all splits author-disjoint

✓ Train authors: 4000 → 248732 docs
✓ Val authors: 1000 (500 in-gallery, 500 OOG)
✓ Test authors: 1000 (500 in-gallery, 500 OOG)


PosixPath('/app/datasets/blog_test/embeddings/scf_embs_blog_test.npz')